In [ ]:
from google.colab import files
uploaded = files.upload()  # select gdp_data.csv when prompted

Saving gdp_data.csv to gdp_data.csv


In [ ]:
# --- installs ---
!pip install plotly pandas --quiet

# --- imports ---
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- load data ---
df = pd.read_csv('gdp_data.csv')

# --- sanity check ---
print(df.shape)        # should be (25, 6)
print(df.head())
print(df.dtypes)
print(df.isnull().sum())  # should all be 0

(25, 6)
   Year  India   China      USA  Brazil  Indonesia
0  2000  477.2  1211.3  10252.3   655.0      165.0
1  2001  493.9  1339.4  10581.8   558.2      160.4
2  2002  523.8  1470.6  10936.4   504.2      195.7
3  2003  618.4  1660.3  11458.2   558.3      234.8
4  2004  721.6  1955.3  12213.7   663.8      256.8
Year           int64
India        float64
China        float64
USA          float64
Brazil       float64
Indonesia    float64
dtype: object
Year         0
India        0
China        0
USA          0
Brazil       0
Indonesia    0
dtype: int64


In [ ]:
# --- transform: wide to long format ---
df_long = df.melt(id_vars='Year',
                   var_name='Country',
                   value_name='GDP_Billion_USD')

print(df_long.head(10))
print(df_long['Country'].unique())  # verify all 5 countries present

# --- chart 1: nominal GDP line chart ---
fig1 = px.line(
    df_long,
    x='Year',
    y='GDP_Billion_USD',
    color='Country',
    title='Nominal GDP 2000–2024 (Current USD Billions)',
    labels={
        'GDP_Billion_USD': 'GDP (USD Billions)',
        'Year': 'Year',
        'Country': 'Country'
    },
    color_discrete_map={
        'India':     '#378ADD',
        'China':     '#E24B4A',
        'USA':       '#639922',
        'Brazil':    '#EF9F27',
        'Indonesia': '#7F77DD'
    }
)

fig1.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#2C2C2A'),
    title_font_size=16,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.02,
        xanchor='right',
        x=1
    ),
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='#F1EFE8', dtick=2),
    yaxis=dict(showgrid=True, gridcolor='#F1EFE8')
)

fig1.update_traces(line=dict(width=2.5))

# highlight India line
fig1.update_traces(
    line=dict(width=4),
    selector=dict(name='India')
)

fig1.show()

   Year Country  GDP_Billion_USD
0  2000   India            477.2
1  2001   India            493.9
2  2002   India            523.8
3  2003   India            618.4
4  2004   India            721.6
5  2005   India            834.2
6  2006   India            949.1
7  2007   India           1238.7
8  2008   India           1224.1
9  2009   India           1365.4
['India' 'China' 'USA' 'Brazil' 'Indonesia']


In [ ]:
# --- compute growth rate ---
countries = ['India', 'China', 'USA', 'Brazil', 'Indonesia']

for c in countries:
    df[f'{c}_growth'] = df[c].pct_change() * 100

# verify
print(df[['Year', 'India_growth', 'China_growth']].head(10))

# --- build growth rate long format ---
growth_cols = [f'{c}_growth' for c in countries]
df_growth = df[['Year'] + growth_cols].copy()
df_growth.columns = ['Year'] + countries
df_growth_long = df_growth.melt(id_vars='Year',
                                 var_name='Country',
                                 value_name='Growth_Rate')

# --- chart 2: growth rate line chart ---
color_map = {
    'India':     '#378ADD',
    'China':     '#E24B4A',
    'USA':       '#639922',
    'Brazil':    '#EF9F27',
    'Indonesia': '#7F77DD'
}

fig2 = px.line(
    df_growth_long,
    x='Year',
    y='Growth_Rate',
    color='Country',
    title='Annual GDP Growth Rate 2000–2024 (%)',
    labels={
        'Growth_Rate': 'Growth Rate (%)',
        'Year': 'Year',
        'Country': 'Country'
    },
    color_discrete_map=color_map
)

# add zero reference line
fig2.add_hline(
    y=0,
    line_dash='dash',
    line_color='#888780',
    line_width=1,
    annotation_text='0%',
    annotation_position='right'
)

# annotate COVID dip for India
fig2.add_annotation(
    x=2020, y=-5.8,
    text='COVID-19<br>India −5.8%',
    showarrow=True,
    arrowhead=2,
    ax=40, ay=-40,
    font=dict(size=11, color='#378ADD'),
    arrowcolor='#378ADD'
)

fig2.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#2C2C2A'),
    title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='#F1EFE8', dtick=2),
    yaxis=dict(showgrid=True, gridcolor='#F1EFE8', ticksuffix='%')
)

fig2.update_traces(line=dict(width=2.5))
fig2.update_traces(line=dict(width=4), selector=dict(name='India'))

fig2.show()

# --- compute per capita ---
# population in millions (World Bank, approximate)
population = {
    'India':     [1056,1071,1086,1102,1118,1134,1150,1166,1182,1198,1214,1230,1247,1263,1280,1296,1313,1330,1347,1364,1380,1393,1407,1429,1442],
    'China':     [1263,1271,1280,1288,1296,1304,1312,1318,1324,1331,1337,1341,1354,1361,1368,1375,1383,1390,1395,1400,1411,1412,1412,1409,1408],
    'USA':       [282,285,288,291,294,297,300,303,306,309,312,314,317,319,322,325,327,330,332,334,331,332,334,336,338],
    'Brazil':    [174,177,180,183,186,189,191,194,197,199,202,204,207,209,212,204,207,209,211,212,213,215,216,217,218],
    'Indonesia': [211,214,217,220,223,226,229,232,235,237,240,243,246,249,252,255,258,261,264,267,270,272,275,278,280]
}

pop_df = pd.DataFrame(population)
pop_df['Year'] = df['Year'].values

for c in countries:
    df[f'{c}_percapita'] = (df[c] * 1e9) / (pop_df[c] * 1e6)

# verify
print(df[['Year', 'India_percapita', 'USA_percapita']].tail(5))

# --- build per capita long format ---
pc_cols = [f'{c}_percapita' for c in countries]
df_pc = df[['Year'] + pc_cols].copy()
df_pc.columns = ['Year'] + countries
df_pc_long = df_pc.melt(id_vars='Year',
                         var_name='Country',
                         value_name='GDP_Per_Capita')

# --- chart 3: per capita line chart ---
fig3 = px.line(
    df_pc_long,
    x='Year',
    y='GDP_Per_Capita',
    color='Country',
    title='GDP Per Capita 2000–2024 (Current USD)',
    labels={
        'GDP_Per_Capita': 'GDP Per Capita (USD)',
        'Year': 'Year',
        'Country': 'Country'
    },
    color_discrete_map=color_map
)

fig3.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#2C2C2A'),
    title_font_size=16,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    hovermode='x unified',
    xaxis=dict(showgrid=True, gridcolor='#F1EFE8', dtick=2),
    yaxis=dict(showgrid=True, gridcolor='#F1EFE8', tickprefix='$')
)

fig3.update_traces(line=dict(width=2.5))
fig3.update_traces(line=dict(width=4), selector=dict(name='India'))

fig3.show()

   Year  India_growth  China_growth
0  2000           NaN           NaN
1  2001      3.499581     10.575415
2  2002      6.053857      9.795431
3  2003     18.060328     12.899497
4  2004     16.688228     17.767873
5  2005     15.604213     16.907891
6  2006     13.773675     20.394593
7  2007     30.513118     29.003307
8  2008     -1.178655     29.405966
9  2009     11.543175     11.044120


    Year  India_percapita  USA_percapita
20  2020      1938.333333   63122.960725
21  2021      2261.521895   70226.204819
22  2022      2382.800284   76235.628743
23  2023      2484.184745   81422.023810
24  2024      2713.384189   85798.816568


In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

color_map = {
    'India':     '#378ADD',
    'China':     '#E24B4A',
    'USA':       '#639922',
    'Brazil':    '#EF9F27',
    'Indonesia': '#7F77DD'
}

# --- build subplot grid ---
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=(
        'Nominal GDP (USD Billions)',
        'GDP Growth Rate (%)',
        'GDP Per Capita (USD)',
        'India GDP Absolute Growth',
        'Share of World GDP (%)',
        'India vs China — Gap Over Time'
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# ── chart 1: nominal GDP (row1, col1) ──
for c in countries:
    fig.add_trace(
        go.Scatter(
            x=df['Year'],
            y=df[c],
            name=c,
            line=dict(color=color_map[c], width=4 if c == 'India' else 2),
            legendgroup=c,
            showlegend=True
        ),
        row=1, col=1
    )

# ── chart 2: growth rate (row1, col2) ──
for c in countries:
    fig.add_trace(
        go.Scatter(
            x=df_growth_long[df_growth_long['Country'] == c]['Year'],
            y=df_growth_long[df_growth_long['Country'] == c]['Growth_Rate'],
            name=c,
            line=dict(color=color_map[c], width=4 if c == 'India' else 2),
            legendgroup=c,
            showlegend=False
        ),
        row=1, col=2
    )

# zero line on growth chart
fig.add_hline(y=0, line_dash='dash', line_color='#888780',
              line_width=1, row=1, col=2)

# ── chart 3: per capita (row2, col1) ──
for c in countries:
    fig.add_trace(
        go.Scatter(
            x=df_pc_long[df_pc_long['Country'] == c]['Year'],
            y=df_pc_long[df_pc_long['Country'] == c]['GDP_Per_Capita'],
            name=c,
            line=dict(color=color_map[c], width=4 if c == 'India' else 2),
            legendgroup=c,
            showlegend=False
        ),
        row=2, col=1
    )

# ── chart 4: India absolute GDP bar (row2, col2) ──
fig.add_trace(
    go.Bar(
        x=df['Year'],
        y=df['India'],
        name='India GDP',
        marker_color='#378ADD',
        opacity=0.85,
        showlegend=False
    ),
    row=2, col=2
)

# ── chart 5: share of world GDP (row3, col1) ──
# world GDP approximation per year (sum of all 5 + rest scaled)
world_gdp = {
    2000:33596, 2001:33601, 2002:34776, 2003:38785, 2004:43084,
    2005:47023, 2006:51956, 2007:58228, 2008:63674, 2009:60236,
    2010:66157, 2011:74106, 2012:75617, 2013:77317, 2014:79563,
    2015:75003, 2016:76378, 2017:81052, 2018:85805, 2019:87752,
    2020:85022, 2021:97834, 2022:101562, 2023:105435, 2024:110000
}

df['World_GDP'] = df['Year'].map(world_gdp)

for c in countries:
    share = (df[c] / df['World_GDP']) * 100
    fig.add_trace(
        go.Scatter(
            x=df['Year'],
            y=share,
            name=c,
            line=dict(color=color_map[c], width=4 if c == 'India' else 2),
            legendgroup=c,
            showlegend=False
        ),
        row=3, col=1
    )

# ── chart 6: India vs China gap (row3, col2) ──
gap = df['China'] - df['India']

fig.add_trace(
    go.Scatter(
        x=df['Year'],
        y=df['China'],
        name='China',
        line=dict(color='#E24B4A', width=2),
        fill=None,
        legendgroup='China',
        showlegend=False
    ),
    row=3, col=2
)

fig.add_trace(
    go.Scatter(
        x=df['Year'],
        y=df['India'],
        name='India',
        line=dict(color='#378ADD', width=2),
        fill='tonexty',
        fillcolor='rgba(55,138,221,0.12)',
        legendgroup='India',
        showlegend=False
    ),
    row=3, col=2
)

# ── global layout ──
fig.update_layout(
    title=dict(
        text='<b>GDP Dashboard — India vs World (2000–2024)</b><br>'
             '<span style="font-size:12px;color:#888">Source: World Bank / IMF · Current USD</span>',
        x=0.5,
        xanchor='center',
        font=dict(size=20)
    ),
    height=1100,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12, color='#2C2C2A'),
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.04,
        xanchor='center',
        x=0.5,
        font=dict(size=12)
    ),
    hovermode='x unified'
)

# clean axes
for i in range(1, 4):
    for j in range(1, 3):
        fig.update_xaxes(showgrid=True, gridcolor='#F1EFE8',
                         dtick=4, row=i, col=j)
        fig.update_yaxes(showgrid=True, gridcolor='#F1EFE8',
                         row=i, col=j)

# y-axis labels
fig.update_yaxes(title_text='USD Billions', row=1, col=1)
fig.update_yaxes(title_text='Growth %', ticksuffix='%', row=1, col=2)
fig.update_yaxes(title_text='USD', tickprefix='$', row=2, col=1)
fig.update_yaxes(title_text='USD Billions', row=2, col=2)
fig.update_yaxes(title_text='% of World GDP', ticksuffix='%', row=3, col=1)
fig.update_yaxes(title_text='USD Billions', row=3, col=2)

fig.show()

# --- export as standalone HTML ---
fig.write_html('gdp_dashboard.html')
print("Saved: gdp_dashboard.html")

# --- download it ---
from google.colab import files
files.download('gdp_dashboard.html')

Saved: gdp_dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

color_map = {
    'India':     '#378ADD',
    'China':     '#E24B4A',
    'USA':       '#639922',
    'Brazil':    '#EF9F27',
    'Indonesia': '#7F77DD'
}

countries = ['India', 'China', 'USA', 'Brazil', 'Indonesia']

# --- compute growth rate ---
for c in countries:
    df[f'{c}_growth'] = df[c].pct_change() * 100

# --- population for per capita ---
population_2024 = {
    'India': 1442, 'China': 1408,
    'USA': 338, 'Brazil': 218, 'Indonesia': 280
}

pc_2024 = {
    c: round((df.loc[df['Year']==2024, c].values[0] * 1e9)
             / (population_2024[c] * 1e6))
    for c in countries
}

sorted_countries = sorted(pc_2024, key=pc_2024.get, reverse=True)
sorted_values    = [pc_2024[c] for c in sorted_countries]
sorted_colors    = [color_map[c] for c in sorted_countries]

# --- layout ---
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.5, 0.5],
    column_widths=[0.55, 0.45],
    subplot_titles=[
        'Nominal GDP 2000–2024 (USD Billions)',
        'Annual GDP Growth Rate (%)',
        'GDP Per Capita 2024 — Comparison (USD)',
        'India GDP — Absolute Growth (USD Billions)'
    ],
    vertical_spacing=0.14,
    horizontal_spacing=0.10
)

# ── chart 1: nominal GDP ──
for c in countries:
    fig.add_trace(
        go.Scatter(
            x=df['Year'],
            y=df[c],
            name=c,
            line=dict(
                color=color_map[c],
                width=4 if c == 'India' else 2,
                dash='solid'
            ),
            legendgroup=c,
            showlegend=True,
            hovertemplate='%{y:.0f}B<extra>' + c + '</extra>'
        ),
        row=1, col=1
    )

# annotations
events = [
    dict(x=2008, y=1224, text='2008 crisis', ay=-45, ax=25),
    dict(x=2020, y=2675, text='COVID',       ay=-45, ax=35),
]
for e in events:
    fig.add_annotation(
        x=e['x'], y=e['y'],
        text=e['text'],
        showarrow=True, arrowhead=2,
        arrowcolor='#378ADD', arrowwidth=1.5,
        ax=e['ax'], ay=e['ay'],
        font=dict(size=11, color='#378ADD'),
        bgcolor='white',
        bordercolor='#B5D4F4',
        borderwidth=1, borderpad=3,
        row=1, col=1
    )

# ── chart 2: growth rate ──
for c in countries:
    fig.add_trace(
        go.Scatter(
            x=df['Year'],
            y=df[f'{c}_growth'],
            name=c,
            line=dict(
                color=color_map[c],
                width=3 if c == 'India' else 1.5
            ),
            legendgroup=c,
            showlegend=False,
            hovertemplate='%{y:.1f}%<extra>' + c + '</extra>'
        ),
        row=1, col=2
    )

fig.add_hline(
    y=0, line_dash='dash',
    line_color='#B4B2A9', line_width=1,
    row=1, col=2
)

# ── chart 3: per capita bar ──
fig.add_trace(
    go.Bar(
        x=sorted_countries,
        y=sorted_values,
        marker_color=sorted_colors,
        text=[f'${v:,}' for v in sorted_values],
        textposition='outside',
        textfont=dict(size=12),
        showlegend=False,
        hovertemplate='%{x}: $%{y:,}<extra></extra>'
    ),
    row=2, col=1
)

# ── chart 4: India absolute bar ──
fig.add_trace(
    go.Bar(
        x=df['Year'],
        y=df['India'],
        marker_color='#378ADD',
        opacity=0.85,
        showlegend=False,
        hovertemplate='%{x}: $%{y:.0f}B<extra>India</extra>'
    ),
    row=2, col=2
)

# ── layout ──
fig.update_layout(
    title=dict(
        text='<b>India GDP — A 25-Year Story (2000–2024)</b>',
        x=0.5,
        xanchor='center',
        font=dict(size=20, color='#2C2C2A')
    ),
    height=800,
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=13, color='#2C2C2A'),
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.12,
        xanchor='center', x=0.5,
        font=dict(size=13),
        tracegroupgap=6
    ),
    hovermode='x unified',
    margin=dict(l=60, r=40, t=80, b=80),
    annotations=[
        dict(
            text='Source: World Bank · IMF · Current USD',
            x=0.5, y=-0.08,
            xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=11, color='#888780'),
            xanchor='center'
        )
    ]
)

# ── axis styling ──
for row, col in [(1,1),(1,2),(2,1),(2,2)]:
    fig.update_xaxes(
        showgrid=True, gridcolor='#F1EFE8',
        linecolor='#D3D1C7', linewidth=0.5,
        row=row, col=col
    )
    fig.update_yaxes(
        showgrid=True, gridcolor='#F1EFE8',
        linecolor='#D3D1C7', linewidth=0.5,
        row=row, col=col
    )

fig.update_xaxes(dtick=4, row=1, col=1)
fig.update_xaxes(dtick=4, row=1, col=2)
fig.update_xaxes(dtick=4, row=2, col=2)

fig.update_yaxes(title_text='USD Billions', tickprefix='$', row=1, col=1)
fig.update_yaxes(title_text='Growth %',     ticksuffix='%', row=1, col=2)
fig.update_yaxes(title_text='USD',          tickprefix='$', row=2, col=1)
fig.update_yaxes(title_text='USD Billions', tickprefix='$', row=2, col=2)

fig.show()

# --- export ---
fig.write_html(
    'gdp_dashboard_clean.html',
    include_plotlyjs='cdn',
    full_html=True,
    config={
        'displayModeBar': True,
        'modeBarButtonsToRemove': ['lasso2d', 'select2d'],
        'toImageButtonOptions': {
            'format': 'png',
            'filename': 'gdp_dashboard_clean',
            'height': 800,
            'width': 1400,
            'scale': 2
        }
    }
)

print("Saved: gdp_dashboard_clean.html")

from google.colab import files
files.download('gdp_dashboard_clean.html')

Saved: gdp_dashboard_clean.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>